# **CS 425 Final Project**
The purpose of this project is to visualize and measure the differences in accuracy and effectiveness between various machine learning architectures and methods. 

For section 1 of the project, we will look at how a standard Convolutional Neural Network (CNN) handles the prediction of read coverage in DNA accessibility versus a transformer architecture such as PDLLM or DNABERT-2.

For section 2 of the project, we will evaluate the performance differences in predicting enhancer activity in Arabidopsis between a standard CNN, a CNN with transfer learning, and a fine-tuned transformer.

The link to our project proposal, timelines, and detailed layout our plans is [here](https://docs.google.com/document/d/1emR3MxvZgMj_qGZ2zw8oQoUpwPpePoVyOWwxj0TpfPU/edit?tab=t.0). Additionally, the download for the dataset we're working with is available on [zenodo](https://zenodo.org/records/19509176) , which will need to be downloaded manually to run the cells, as the folder is too big to upload to Github without LFS.

## DNA Accessibility ##

#### Data Pre-Processing ####

In [ ]:
import pyBigWig
import numpy as np
import os
from multiprocessing import Pool

winsize = 2000
windows = []

# original code + modifications from claude to support multithreading
for file in os.scandir("data/new_comb_data"):
    if file.name.endswith(".bw"):
        bw = pyBigWig.open(file.path)
        for chr in bw.chroms():
            for start in range(0, bw.chroms(chr) - winsize + 1, winsize):
                windows.append((chr, start, start + winsize))
        bw.close()
        break

def process_file(filepath):
    print(f"Processing {os.path.basename(filepath)} ...")
    bw = pyBigWig.open(filepath)
    exp = []
    for chrom, start, end in windows:
        center       = (start + end) // 2
        center_start = center - 100
        center_end   = center + 100
        bins = bw.stats(chrom, center_start, center_end, type="mean", nBins=10)
        exp.append([b if b is not None else 0.0 for b in bins])
    bw.close()
    return exp

filepaths = [f.path for f in os.scandir("data/new_comb_data") if f.name.endswith(".bw")]

print(f"Processing {len(filepaths)} files across {os.cpu_count()} cores ...")

with Pool(processes=os.cpu_count()) as pool:
    data = pool.map(process_file, filepaths)

Y = np.stack(data, axis=1)
Y = np.log1p(Y)

print(Y.shape)

Processing 64 files across 12 cores ...
Processing SRX1096549_Rep0.rpgc.bw ...Processing SRX1096551_Rep0.rpgc.bw ...Processing DRX092570.bw ...Processing SRX130305.bw ...Processing SRX1098138_Rep0.rpgc.bw ...Processing SRX2467594.bw ...Processing SRX2240741.bw ...Processing SRX2140205.bw ...Processing SRX215438.bw ...Processing SRX1412761.bw ...Processing SRX1098136_Rep0.rpgc.bw ...
Processing SRX1012884.bw ...










Processing SRX1096548_Rep0.rpgc.bw ...
Processing SRX3152333.bw ...
Processing SRX2311142.bw ...
Processing SRX1098137_Rep0.rpgc.bw ...
Processing SRX1098135_Rep0.rpgc.bw ...
Processing SRX215433.bw ...
Processing SRX1161361.bw ...
Processing SRX2140200.bw ...
Processing SRX1096550_Rep0.rpgc.bw ...
Processing SRX215474.bw ...
Processing SRX1012869.bw ...
Processing SRX130311.bw ...
Processing SRX3753856.bw ...
Processing SRX391990_Rep0.rpgc.bw ...
Processing SRX391992_Rep0.rpgc.bw ...
Processing SRX391994_Rep0.rpgc.bw ...
Processing SRX391996_Rep0.rpgc.bw ...
Processin

In [ ]:
import pyBigWig
import numpy as np
import os

windows = []
data = []

with os.scandir("data/new_comb_data") as files:
    winsize = 2000
    for file in files:
        start = 0
        if not file.name.endswith(".bw"):
            continue
        bw = pyBigWig.open(file.path)
        chroms = bw.chroms()
        
        if (len(windows) == 0):
            for chr in bw.chroms():
                for start in range(0, bw.chroms(chr) - winsize + 1, winsize):
                    windows.append((chr, start, start + winsize))
                    
        exp = []
        # print(chroms)
        for chr, start, end in windows:
            center = (start + end) // 2
            bins = bw.stats(chr, (center - 100), (center + 100), type="mean", nBins = 10)
            exp.append([b if b is not None else 0.0 for b in bins])
            # print(bins)
        bw.close()
        
        data.append(exp)
        
Y = np.stack(data, axis=1)
Y = np.log1p(Y)

print(Y.shape)

### **Architectures** ###

#### Basset (Multi-layer CNN) ####

#### Transformer ####

### **Analysis of Results** ###

## Enhancer Activity in Arabidopsis ##

#### Data Pre-Processing ####

### **Architectures** ###

#### Multi-layer CNN without Transfer Learning ####

#### Multi-layer CNN with Transfer Learning ####

#### Fine-tuned Transformer ####

### **Analysis of Results** ###

# **Conclusion** #